In [35]:
from pathlib import Path
import os
import sys
from dotenv import load_dotenv
from groq import Groq
import json
import importlib
importlib.reload(server)

<module 'server' from 'c:\\Users\\vitor\\OneDrive - Indev\\Estudos\\estudos_python\\2_estudos_mcp\\mcp_anomalias_reclamacao\\teste\\mcp_teste\\server.py'>

In [36]:
env_path = Path(r"C:\Users\vitor\OneDrive - Indev\Estudos\estudos_python\2_estudos_mcp\mcp_anomalias_reclamacao\.env")
server_path = Path(r"C:\Users\vitor\OneDrive - Indev\Estudos\estudos_python\2_estudos_mcp\mcp_anomalias_reclamacao\teste\mcp_teste")

load_dotenv(env_path)

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("GROQ_API_KEY não encontrada no arquivo .env")

client = Groq(api_key=api_key)

if str(server_path) not in sys.path:
    sys.path.append(str(server_path))

print("Groq carregada e server MCP importado com sucesso.")

Groq carregada e server MCP importado com sucesso.


In [37]:
# ── 1. Definição das tools no formato Groq/OpenAI ────────────────────────────
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "skill_teste",
            "description": "Skill de teste com soma de 3 valores para validar o fluxo de orquestração e cadastro de skills.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer", "description": "Primeiro valor"},
                    "b": {"type": "integer", "description": "Segundo valor"},
                    "c": {"type": "integer", "description": "Terceiro valor"},
                    "d": {"type": "integer", "description": "Quarto valor"}
                },
                "required": ["a", "b", "c", "d"],
            },
        },
    }
]

TOOL_MAP = {
    "skill_teste": server.skill_teste,
}

# ── 2. Carrega system prompt do arquivo .md ───────────────────────────────────
prompt_path = server_path / "prompts" / "1_orquestrador.md"
SYSTEM_PROMPT = prompt_path.read_text(encoding="utf-8")

# ── 3. Loop de orquestração ───────────────────────────────────────────────────
def orquestrar(user_message: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]

    while True:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
        )

        choice = response.choices[0]

        # Sem tool call → resposta final
        if choice.finish_reason != "tool_calls":
            return choice.message.content

        # Processa tool calls
        tool_calls = choice.message.tool_calls
        messages.append(choice.message)

        for tc in tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            print(f"  [tool call] {fn_name}({fn_args})")

            resultado = TOOL_MAP[fn_name](**fn_args)

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(resultado),
            })

In [40]:
# ── 4. Teste ─────────────────────────────────────────────────────────────────
resposta = orquestrar("qual a soma dos números 10, 20, 30")
print(f"\n[Resposta final]\n{resposta}")


  [tool call] skill_teste({'a': 10, 'b': 20, 'c': 30, 'd': 0})

[Resposta final]
O resultado da soma de 10, 20 e 30 é 60.
